Genetic status cohort definition analysis in GP2 Neurobooster genotyping data (all ancestries)

Project: GP2 lysosomal PRS

Version: Python/3.10.17, R/4.4.2

Notebook Overview

1. Description Loading Python libraries Set paths Make working directory

2. Installing packages

3. Process clinical data

4. GBA1 and LRRK2 cohort definition (shared by GP2 consortium)

5. Known pathogenic genes cohort definition

6. Cohort sorting

Loading Python libraries

In [1]:
# Use pathlib for file path manipulation
import pathlib

# Install numpy
import numpy as np

# Install Pandas for tabular data
import pandas as pd

# Install plotnine: a ggplot2-compatible Python plotting package
from plotnine import *

# Always show all columns in a Pandas DataFrame
pd.set_option('display.max_columns', None)

Set paths

In [2]:
WORK_DIR = "~/workspace/ws_files/"

In [3]:
REL11_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release11')
!ls -hal {REL11_PATH}

total 21K
dr-xr-xr-x. 1 jupyter users   0 May 19 09:28 clinical_data
dr-xr-xr-x. 1 jupyter users   0 May 19 09:49 clinical_exomes
dr-xr-xr-x. 1 jupyter users   0 May 19 09:49 imputed_genotypes
dr-xr-xr-x. 1 jupyter users   0 May 19 09:49 meta_data
dr-xr-xr-x. 1 jupyter users   0 May 19 09:49 raw_genotypes
dr-xr-xr-x. 1 jupyter users   0 May 19 09:49 raw_genotypes_flipped
-r--r--r--. 1 jupyter users 21K Dec  2 04:24 README_R11.txt
dr-xr-xr-x. 1 jupyter users   0 May 19 09:49 wgs


install packages

PLINK

In [4]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink; then
echo "Plink1.9 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in /home/jupyter/tools \n "

fi

    -------


--2025-08-08 13:23:03--  http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.112.8, 52.216.36.216, 16.15.193.113, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.112.8|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8708135 (8.3M) [application/zip]
Saving to: ‘plink_linux_x86_64_20190304.zip’

     0K .......... .......... .......... .......... ..........  0%  293K 29s
    50K .......... .......... .......... .......... ..........  1%  585K 22s
   100K .......... .......... .......... .......... ..........  1% 58.8M 14s
   150K .......... .......... .......... .......... ..........  2%  251M 11s
   200K .......... .......... .......... .......... ..........  2%  589K 11s
   250K .......... .......... .......... .......... ..........  3%  100M 9s
   300K .......... .......... .......... .......... ..........  4%  235M 8s
   350K .......... .......... .......... .......... ....

Archive:  plink_linux_x86_64_20190304.zip
  inflating: plink                   
  inflating: LICENSE                 
  inflating: toy.ped                 
  inflating: toy.map                 
  inflating: prettify                

 plink downloaded and unzipped in /home/jupyter/tools 
 


In [5]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/tools/
if test -e /home/jupyter/tools/plink2; then

echo "Plink2 is already installed in /home/jupyter/tools/"
else
echo "Plink2 is not installed"
cd /home/jupyter/tools/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [35]:
%%bash

# Install ANNOVAR:
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then

echo "annovar is already installed in /home/jupyter/tools/"
else
echo "annovar is not installed"
cd /home/jupyter/tools/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

annovar is already installed in /home/jupyter/tools/


Install ANNOVAR: Download sources of annotation

In [36]:
%%bash

cd /home/jupyter/tools/annovar/

perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20140902 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ensGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp147 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp30a humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad211_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ljb26_all humandb/

NOTICE: Web-based checking to see whether ANNOVAR new version is available ... Done
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_refGene.txt.gz ... OK
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_refGeneMrna.fa.gz ... OK
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_refGeneVersion.txt.gz ... OK
NOTICE: Uncompressing downloaded files
NOTICE: Finished downloading annotation files for hg38 build version, with files saved at the 'humandb' directory
NOTICE: Web-based checking to see whether ANNOVAR new version is available ... Done
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_clinvar_20140902.txt.gz ... OK
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_clinvar_20140902.txt.idx.gz ... OK
NOTICE: Uncompressing downloaded files
NOTICE: Finished d

In [4]:
! ls /home/jupyter/tools

annovar				       plink2_linux_x86_64_latest.zip
annovar.latest.tar.gz		       plink_linux_x86_64_20190304.zip
intel-simplified-software-license.txt  prettify
LICENSE				       toy.map
plink				       toy.ped
plink2				       vcf_subset


process clinical data

In [4]:
CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')

In [ ]:
# Let's load the master key
key = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(key.shape)
key.head()

In [17]:
# look at sample size of each cohort

(key['study'] == 'TUEPAC').sum()

3706

In [ ]:
# filter out european population firstly
EUR = key[key['nba_label'] == 'EUR']
EUR

In [ ]:
# filter out european population firstly
FIN = key[key['nba_label'] == 'FIN']
FIN

In [ ]:
# Subsetting to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)
key

In [11]:
%%bash
# making working directory
#Loop over all the ancestries
for ancestry in {'EUR','AAC','AFR','AJ','AMR','CAS','EAS','MDE','SAS','CAH'} ;
do

#Make a folder for each ancestry
mkdir ~/workspace/ws_files/r11/cohort/"$ancestry"

done

In [15]:
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'
    print(f'WORKING ON: {ancestry}')
    
    ## Subset to keep ancestry of interest 
    ancestry_key = key[key['nba_label']==ancestry].copy()
    ancestry_key.reset_index(drop=True)
    
     # Load information about related individuals in the ancestry analyzed
    related_df = pd.read_csv(f'{REL11_PATH}/meta_data/related_samples/{ancestry}_release11_vwb.related')
    print(f'Related individuals: {related_df.shape}')
    
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])
    
    # Check value counts of related and remove only one related individual
    ancestry_key = ancestry_key[~ancestry_key["IID"].isin(related_list)]
    
    # Check size
    print(f'Unrelated individuals: {ancestry_key.shape}')
    
    # Convert phenotype to binary (1/2)
    ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
    # PD = 2; control = 1
    pheno_mapping = {"PD": 2, "Control": 1}
    ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')
    
    # Check value counts of pheno
    ancestry_key['PHENO'].value_counts(dropna=False)
    
    # Check value counts of SEX
    sex_og_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
    
     # Convert sex to binary (1/2)
    ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
    # Female = 2; Male = 1
    sex_mapping = {"Female": 2, "Male": 1}
    ancestry_key['SEX'] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')
    
    # Check value counts of SEX after recoding
    sex_recode_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
    
    #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
    #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
    #Uppercase column name is ok
    #See https://zhanxw.github.io/rvtests/#phenotype-file
    ancestry_key['FID'] = 0
    ancestry_key['FATID'] = 0
    ancestry_key['MATID'] = 0
    
    ## Clean up and keep columns we need 
    final_df = ancestry_key[['FID','IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO']].copy()
    final_df['FID'] = final_df['IID']
    
    ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
    #Leave missing values as NA
    
    #Check number of PD cases missing age
    pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
    print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
    
    #Check number of controls missing age
    control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
    print(f'Number of controls missing age: {control_missAge.shape[0]}')
    
    ## Make file of sample IDs to keep 
    samples_toKeep = final_df[['FID', 'IID']].copy()
    samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/{ancestry}/{ancestry}.samplestoKeep', sep = '\t', index=False, header=None)
    
    ## Make your covariate file
    #Included na_rep to write out missing/NA values explicitly as string/text, not as blank otherwise they are misread in RVtests
    final_df.to_csv(f'~/workspace/ws_files/r11/cohort/{ancestry}/{ancestry}_covariate_file.txt', sep = '\t', na_rep='NA', index=False)

WORKING ON: AAC
Related individuals: (12, 9)
Unrelated individuals: (1490, 6)
Sex value counts - original:
 SEX
Female                        811
Male                          677
Other/Unknown/Not Reported      2
Sex value counts - recoded:
SEX
2       811
1       677
<NA>      2
Number of PD cases missing age: 86
Number of controls missing age: 24
WORKING ON: SAS
Related individuals: (121, 9)
Unrelated individuals: (1306, 6)
Sex value counts - original:
 SEX
Male      840
Female    466
Sex value counts - recoded:
SEX
1    840
2    466
Number of PD cases missing age: 63
Number of controls missing age: 88
WORKING ON: AFR
Related individuals: (429, 9)
Unrelated individuals: (7830, 6)
Sex value counts - original:
 SEX
Male                          4602
Female                        3226
Other/Unknown/Not Reported       2
Sex value counts - recoded:
SEX
1       4602
2       3226
<NA>       2
Number of PD cases missing age: 158
Number of controls missing age: 49
WORKING ON: CAS
Related ind

GBA1 cohort definition

Lara's file to define GBA1 cohort

In [ ]:
# read gba1_lrrk2 ccarriers file
carriers_ancestry = pd.read_csv("~/workspace/ws_files/r11/results/GBA1_LRRK2_check/carriers_ancestry.csv", sep=",")
carriers_ancestry

In [ ]:
# keep GBA1 risk carriers
GBA1_risk = carriers_ancestry[
    carriers_ancestry['carrier_group'].isin(['GBA1_only', 'GBA1_LRRK2'])
].copy()
GBA1_risk

In [ ]:
# remove missing ancestry or GP2ID
GBA1_risk = GBA1_risk.dropna(subset=['Ancestry', 'GP2ID'])
GBA1_risk

In [ ]:
# clean dataframe
GBA1_risk_clean = GBA1_risk.copy()

# remove old index column if present
GBA1_risk_clean = GBA1_risk_clean.drop(columns=['Unnamed: 0'], errors='ignore')

# remove missing GP2ID or Ancestry
GBA1_risk_clean = GBA1_risk_clean.dropna(subset=['GP2ID', 'Ancestry'])

# remove duplicated GP2ID within each ancestry
GBA1_risk_clean = GBA1_risk_clean.drop_duplicates(subset=['Ancestry', 'GP2ID'])

GBA1_risk_clean

In [10]:
GBA1_risk_clean['Ancestry'].value_counts()

Ancestry
EUR    5664
AFR    3213
AJ      928
AAC     419
EAS     241
AMR     206
CAH     203
CAS     109
MDE      98
SAS      52
FIN      35
Name: count, dtype: int64

In [12]:
import os
# base output directory
base_dir = os.path.expanduser("~/workspace/ws_files/r11/cohort")

# save one file for each ancestry
for ancestry, df_anc in GBA1_risk_clean.groupby('Ancestry'):

    # create ancestry folder
    out_dir = os.path.join(base_dir, ancestry)
    os.makedirs(out_dir, exist_ok=True)

    # make FID and IID columns from GP2ID
    samples_toKeep = pd.DataFrame({
        'FID': df_anc['GP2ID'],
        'IID': df_anc['GP2ID']
    })

    # output file
    out_file = os.path.join(out_dir, "GBA1risk.samplestoKeep.rm.txt")

    # save without header and without index
    samples_toKeep.to_csv(
        out_file,
        sep='\t',
        index=False,
        header=False
    )

    print(f"{ancestry}: {samples_toKeep.shape[0]} samples saved to {out_file}")

AAC: 419 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AAC/GBA1risk.samplestoKeep.rm.txt
AFR: 3213 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AFR/GBA1risk.samplestoKeep.rm.txt
AJ: 928 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AJ/GBA1risk.samplestoKeep.rm.txt
AMR: 206 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AMR/GBA1risk.samplestoKeep.rm.txt
CAH: 203 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/CAH/GBA1risk.samplestoKeep.rm.txt
CAS: 109 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/CAS/GBA1risk.samplestoKeep.rm.txt
EAS: 241 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/EAS/GBA1risk.samplestoKeep.rm.txt
EUR: 5664 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/EUR/GBA1risk.samplestoKeep.rm.txt
FIN: 35 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/FIN/GBA1risk.samplestoKeep.rm.txt
MDE: 98 samples saved to /home/jupyter/workspace/ws_files/r11/coh

LRRK2 cohort definition

Lara's file to define LRRK2 cohort

In [ ]:
# keep LRRK2 risk carriers
LRRK2_risk = carriers_ancestry[
    carriers_ancestry['carrier_group'].isin(['LRRK2_only', 'GBA1_LRRK2'])
].copy()
LRRK2_risk

In [ ]:
# remove missing ancestry or GP2ID
LRRK2_risk = LRRK2_risk.dropna(subset=['Ancestry', 'GP2ID'])
LRRK2_risk

In [ ]:
# clean dataframe
LRRK2_risk_clean = LRRK2_risk.copy()

# remove old index column if present
LRRK2_risk_clean = LRRK2_risk_clean.drop(columns=['Unnamed: 0'], errors='ignore')

# remove missing GP2ID or Ancestry
LRRK2_risk_clean = LRRK2_risk_clean.dropna(subset=['GP2ID', 'Ancestry'])

# remove duplicated GP2ID within each ancestry
LRRK2_risk_clean = LRRK2_risk_clean.drop_duplicates(subset=['Ancestry', 'GP2ID'])

LRRK2_risk_clean

In [16]:
LRRK2_risk_clean['Ancestry'].value_counts()

Ancestry
EUR    1013
AJ      890
EAS     861
MDE      99
AMR      70
CAH      48
CAS      43
AFR      10
AAC       6
FIN       1
SAS       1
Name: count, dtype: int64

In [17]:
import os
# base output directory
base_dir = os.path.expanduser("~/workspace/ws_files/r11/cohort")

# save one file for each ancestry
for ancestry, df_anc in LRRK2_risk_clean.groupby('Ancestry'):

    # create ancestry folder
    out_dir = os.path.join(base_dir, ancestry)
    os.makedirs(out_dir, exist_ok=True)

    # make FID and IID columns from GP2ID
    samples_toKeep = pd.DataFrame({
        'FID': df_anc['GP2ID'],
        'IID': df_anc['GP2ID']
    })

    # output file
    out_file = os.path.join(out_dir, "LRRK2risk.samplestoKeep.rm.txt")

    # save without header and without index
    samples_toKeep.to_csv(
        out_file,
        sep='\t',
        index=False,
        header=False
    )

    print(f"{ancestry}: {samples_toKeep.shape[0]} samples saved to {out_file}")

AAC: 6 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AAC/LRRK2risk.samplestoKeep.rm.txt
AFR: 10 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AFR/LRRK2risk.samplestoKeep.rm.txt
AJ: 890 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AJ/LRRK2risk.samplestoKeep.rm.txt
AMR: 70 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/AMR/LRRK2risk.samplestoKeep.rm.txt
CAH: 48 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/CAH/LRRK2risk.samplestoKeep.rm.txt
CAS: 43 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/CAS/LRRK2risk.samplestoKeep.rm.txt
EAS: 861 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/EAS/LRRK2risk.samplestoKeep.rm.txt
EUR: 1013 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/EUR/LRRK2risk.samplestoKeep.rm.txt
FIN: 1 samples saved to /home/jupyter/workspace/ws_files/r11/cohort/FIN/LRRK2risk.samplestoKeep.rm.txt
MDE: 99 samples saved to /home/jupyter/workspace/ws_files/r11/co

SNCA cohort definition

Annotation of the gene

Extract the region using PLINK

Extract SNCA gene in NBA cohort(excluding tuepac)

SNCA coordinates: Chromosome 4: 89,700,345-89,838,315(GRCh38/hg38)

In [3]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr4_{ancestry}_release11_vwb \
    --chr 4 \
    --from-bp 89700345 \
    --to-bp 89838315 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_SNCA

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SNCA.log.
Options in effect:
  --chr 4
  --from-bp 89700345
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SNCA
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr4_CAS_release11_vwb
  --to-bp 89838315

Start time: Fri Jan  9 09:14:06 2026
419000 MiB RAM detected, ~414557 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr4_CAS_release11_vwb.psam.
3299157 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr4_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 controls

PRKN cohort definition

Annotation of the gene

Extract the region using PLINK

Extract PRKN gene in NBA cohort(excluding tuepac)

PRKN coordinates: Chromosome 6: 161,347,417-162,727,775(GRCh38/hg38)

In [5]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr6_{ancestry}_release11_vwb \
    --chr 6 \
    --from-bp 161347417 \
    --to-bp 162727775 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_PRKN

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PRKN.log.
Options in effect:
  --chr 6
  --from-bp 161347417
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PRKN
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr6_CAS_release11_vwb
  --to-bp 162727775

Start time: Fri Jan  9 09:19:13 2026
419000 MiB RAM detected, ~414290 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr6_CAS_release11_vwb.psam.
2945424 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr6_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 contro

In [6]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_PRKN \
    --extract ~/workspace/ws_files/cohort/variants.list/PRKN.txt \
    --make-bed \
    --out {WORK_DIR}/PRKN

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/PRKN.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PRKN
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/PRKN.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/PRKN

Start time: Fri Jan  9 09:21:29 2026
419000 MiB RAM detected, ~413584 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PRKN.fam.
26903 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PRKN.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 7 variants remaining.
7 variants remaining after main filters.
done.ng /home/j

In [7]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/PRKN \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/PRKN.txt \
    --recode A \
    --out {WORK_DIR}/PRKN

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/PRKN.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/PRKN
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/PRKN.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/PRKN
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
7 variants loaded from .bim file.
2801 people (1199 males, 1602 females) loaded from .fam.
2424 phenotype values loaded from .fam.
--extract: 7 variants remaining.
--keep: 2416 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2416 founders and 0 nonfounders present.
Calculating allele frequencies... done.
Total genotyping rate in remaining sample

In [8]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
recode = pd.read_csv(f'{WORK_DIR}/CAH/PRKN.raw', sep='\s+')
recode

In [46]:
recode_PRKN = recode.copy()
# Define the list of prkn variant column names
PRKN_variants = [
    'chr6:161350125:T:G_G',
    'chr6:161350208:C:T_T',
    'chr6:161785820:G:A_A',
    'chr6:161973317:G:A_A',
    'chr6:162443325:AT:A_A',
    'chr6:162443378:CCT:C_C'
]

# Add the PRKN_status column
recode_PRKN['PRKN_status'] = recode_PRKN[PRKN_variants].apply(
    lambda row: 'PRKNcarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [47]:
# Filter for PRKN risk carriers
PRKN_carriers = recode_PRKN[recode_PRKN['PRKN_status'] == 'PRKNcarriers']
# Count PHENOTYPE values
PRKN_carriers['PHENOTYPE'].value_counts()

PHENOTYPE
2    11
1     4
Name: count, dtype: int64

In [48]:
# save sample ID of risk carriers
samples_toKeep = PRKN_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/CAH/PRKN.samplestoKeep.txt', sep = '\t', index=False, header=None)

RAB32 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract RAB32 gene in NBA cohort(excluding tuepac)

RAB32 coordinates: Chromosome 6: 146,543,833-146,554,953(GRCh38/hg38)

In [49]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr6_{ancestry}_release11_vwb \
    --chr 6 \
    --from-bp 146543833 \
    --to-bp 146554953 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_RAB32

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB32.log.
Options in effect:
  --chr 6
  --from-bp 146543833
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB32
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr6_CAS_release11_vwb
  --to-bp 146554953

Start time: Fri Jan  9 09:39:34 2026
419000 MiB RAM detected, ~414154 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr6_CAS_release11_vwb.psam.
2945424 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr6_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 cont

In [50]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_RAB32 \
    --extract ~/workspace/ws_files/cohort/variants.list/RAB32.txt \
    --make-bed \
    --out {WORK_DIR}/RAB32

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/RAB32.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB32
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/RAB32.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/RAB32

Start time: Fri Jan  9 09:40:46 2026
419000 MiB RAM detected, ~414190 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB32.fam.
176 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB32.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End

PINK1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract PINK1 gene in NBA cohort(excluding tuepac)

PINK1 coordinates: Chromosome 1: 20,633,458-20,651,511(GRCh38/hg38)

In [51]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 20633458 \
    --to-bp 20651511 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_PINK1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PINK1.log.
Options in effect:
  --chr 1
  --from-bp 20633458
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PINK1
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb
  --to-bp 20651511

Start time: Fri Jan  9 09:42:43 2026
419000 MiB RAM detected, ~414164 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.psam.
3776191 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 contro

In [52]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_PINK1 \
    --extract ~/workspace/ws_files/cohort/variants.list/PINK1.txt \
    --make-bed \
    --out {WORK_DIR}/PINK1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/PINK1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PINK1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/PINK1.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/PINK1

Start time: Fri Jan  9 09:44:02 2026
419000 MiB RAM detected, ~414075 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PINK1.fam.
385 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_PINK1.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 1 variant remaining.
1 variant remaining after main filters.
done.ng /home

In [53]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'CAS','AMR','EUR','EAS','MDE','AFR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/PINK1 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/PINK1.txt \
    --recode A \
    --out {WORK_DIR}/PINK1

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/EAS/PINK1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/EAS/PINK1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/PINK1.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/EAS/EAS.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/EAS/PINK1
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
3 variants loaded from .bim file.
7965 people (4637 males, 3328 females) loaded from .fam.
5870 phenotype values loaded from .fam.
--extract: 3 variants remaining.
--keep: 6346 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 6346 founders and 0 nonfounders present.
Calculating allele frequencies... done.
Total genotyping rate in remaining sa

In [54]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
recode = pd.read_csv(f'{WORK_DIR}/AFR/PINK1.raw', sep='\s+')
recode

In [82]:
recode_PINK1 = recode.copy()
# Define the list of PINK1 variant column names
PINK1_variants = [
    'chr1:20649217:C:T_T'
]

# Add the PINK1_status column
recode_PINK1['PINK1_status'] = recode_PINK1[PINK1_variants].apply(
    lambda row: 'PINK1carriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [83]:
# Filter for pink1 carriers
PINK1_carriers = recode_PINK1[recode_PINK1['PINK1_status'] == 'PINK1carriers']
# Count PHENOTYPE values
PINK1_carriers['PHENOTYPE'].value_counts()

PHENOTYPE
1    1
Name: count, dtype: int64

In [84]:
# save sample ID of risk carriers
samples_toKeep = PINK1_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/PINK1.samplestoKeep.txt', sep = '\t', index=False, header=None)

VPS35 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract VPS35 gene in NBA cohort(excluding tuepac)

VPS35 coordinates: Chromosome 16: 46,656,132-46,689,518 (GRCh38/hg38)

In [85]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f"~/workspace/ws_files/r11/cohort/{ancestry}"

    pfile_path = f"{REL11_PATH}/imputed_genotypes/{ancestry}/chr16_{ancestry}_release11_vwb"
    output_path = f"{WORK_DIR}/{ancestry}_VPS35"

    plink_cmd = f"""
    /home/jupyter/tools/plink2 \
    --pfile {pfile_path} \
    --chr 16 \
    --from-bp 46656132 \
    --to-bp 46689518 \
    --make-bed \
    --out {output_path}
    """

    print(f"Running for ancestry: {ancestry}")
    !{plink_cmd}

Running for ancestry: CAS
PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS35.log.
Options in effect:
  --chr 16
  --from-bp 46656132
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS35
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr16_CAS_release11_vwb
  --to-bp 46689518

Start time: Fri Jan  9 09:55:59 2026
419000 MiB RAM detected, ~414078 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr16_CAS_release11_vwb.psam.
1509705 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr16_CAS_release11_vwb.pvar.
1 binary phenotype l

In [86]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_VPS35 \
    --extract ~/workspace/ws_files/cohort/variants.list/VPS35.txt \
    --make-bed \
    --out {WORK_DIR}/VPS35

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/VPS35.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS35
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/VPS35.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/VPS35

Start time: Fri Jan  9 09:57:18 2026
419000 MiB RAM detected, ~414090 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS35.fam.
331 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS35.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End

DJ1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract DJ1 gene in NBA cohort(excluding tuepac)

DJ1 coordinates: Chromosome 1: 7,954,291-7,985,505 (GRCh38/hg38)

In [87]:
WORK_DIR = '~/workspace/ws_files/r11/cohort/'

In [88]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 7954291 \
    --to-bp 7985505 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_DJ1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DJ1.log.
Options in effect:
  --chr 1
  --from-bp 7954291
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DJ1
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb
  --to-bp 7985505

Start time: Fri Jan  9 09:59:11 2026
419000 MiB RAM detected, ~414081 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.psam.
3776191 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 controls).
6

In [89]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_DJ1 \
    --extract ~/workspace/ws_files/cohort/variants.list/DJ1.txt \
    --make-bed \
    --out {WORK_DIR}/DJ1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/DJ1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DJ1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/DJ1.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/DJ1

Start time: Fri Jan  9 10:00:53 2026
419000 MiB RAM detected, ~414049 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DJ1.fam.
614 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DJ1.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End time: Fri J

In [90]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'EAS','SAS','AFR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/DJ1 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/DJ1.txt \
    --recode A \
    --out {WORK_DIR}/DJ1

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/EAS/DJ1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/EAS/DJ1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/DJ1.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/EAS/EAS.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/EAS/DJ1
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
1 variant loaded from .bim file.
7965 people (4637 males, 3328 females) loaded from .fam.
5870 phenotype values loaded from .fam.
--extract: 1 variant remaining.
--keep: 6346 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 6346 founders and 0 nonfounders present.
Calculating allele frequencies... done.
Total genotyping rate in remaining samples is 0

In [91]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# DJ1
recode = pd.read_csv(f'{WORK_DIR}/AFR/DJ1.raw', sep='\s+')
recode

In [101]:
recode_DJ1 = recode.copy()
# Define the list of PINK1 variant column names
DJ1_variants = [
    'chr1:7962868:G:A_A'
]

# Add the DJ1_status column
recode_DJ1['DJ1_status'] = recode_DJ1[DJ1_variants].apply(
    lambda row: 'DJ1carriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [102]:
# Filter for pink1 carriers
DJ1_carriers = recode_DJ1[recode_DJ1['DJ1_status'] == 'DJ1carriers']
# Count PHENOTYPE values
DJ1_carriers['PHENOTYPE'].value_counts()

PHENOTYPE
1    1
Name: count, dtype: int64

In [103]:
# save sample ID of risk carriers
samples_toKeep = DJ1_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/DJ1.samplestoKeep.txt', sep = '\t', index=False, header=None)

ATP13A2 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract ATP13A2 gene in NBA cohort(excluding tuepac)

ATP13A2 coordinates: Chromosome 1: 16,985,958-17,011,928 (GRCh38/hg38)

In [104]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 16985958 \
    --to-bp 17011928 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_ATP

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_ATP.log.
Options in effect:
  --chr 1
  --from-bp 16985958
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_ATP
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb
  --to-bp 17011928

Start time: Fri Jan  9 10:07:44 2026
419000 MiB RAM detected, ~414108 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.psam.
3776191 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 controls).

In [105]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_ATP \
    --extract ~/workspace/ws_files/cohort/variants.list/ATP13A2.txt \
    --make-bed \
    --out {WORK_DIR}/ATP

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/ATP.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_ATP
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/ATP13A2.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/ATP

Start time: Fri Jan  9 11:52:06 2026
419000 MiB RAM detected, ~414188 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_ATP.fam.
464 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_ATP.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 1 variant remaining.
1 variant remaining after main filters.
done.ng /home/jupyter

In [106]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'CAS','CAH','EUR','AFR','AJ','AMR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/ATP \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/ATP13A2.txt \
    --recode A \
    --out {WORK_DIR}/ATP

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAH/ATP.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAH/ATP
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/ATP13A2.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/CAH/CAH.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAH/ATP
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
1 variant loaded from .bim file.
1268 people (663 males, 605 females) loaded from .fam.
1182 phenotype values loaded from .fam.
--extract: 1 variant remaining.
--keep: 1212 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 1212 founders and 0 nonfounders present.
Calculating allele frequencies... done.
1 variant and 1212 people pass filters and QC

In [107]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# ATP13A2
recode = pd.read_csv(f'{WORK_DIR}/AFR/ATP.raw', sep='\s+')
recode

In [129]:
recode_ATP = recode.copy()
# Define the list of ATP13A2 variant column names
ATP_variants = [
    'chr1:16988455:C:T_T',
    'chr1:16989886:C:T_T'
]
# Add the ATP_status column
recode_ATP['ATP_status'] = recode_ATP[ATP_variants].apply(
    lambda row: 'ATPcarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [130]:
# Filter for ATP risk carriers
ATP_carriers = recode_ATP[recode_ATP['ATP_status'] == 'ATPcarriers']
# Count PHENOTYPE values
ATP_carriers['PHENOTYPE'].value_counts()

PHENOTYPE
1    1
2    1
Name: count, dtype: int64

In [131]:
# save sample ID of risk carriers
samples_toKeep = ATP_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/ATP.samplestoKeep.txt', sep = '\t', index=False, header=None)

DCTN1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract DCTN1 gene in NBA cohort(excluding tuepac)

DCTN1 coordinates: Chromosome 2: 74,361,154-74,392,087 (GRCh38/hg38)

In [132]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr2_{ancestry}_release11_vwb \
    --chr 2 \
    --from-bp 74361154 \
    --to-bp 74392087 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_DCTN1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DCTN1.log.
Options in effect:
  --chr 2
  --from-bp 74361154
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DCTN1
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr2_CAS_release11_vwb
  --to-bp 74392087

Start time: Fri Jan  9 12:03:20 2026
419000 MiB RAM detected, ~414111 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr2_CAS_release11_vwb.psam.
4040601 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr2_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 contro

In [133]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_DCTN1 \
    --extract ~/workspace/ws_files/cohort/variants.list/DCTN1.txt \
    --make-bed \
    --out {WORK_DIR}/DCTN1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/DCTN1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DCTN1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/DCTN1.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/DCTN1

Start time: Fri Jan  9 12:05:16 2026
419000 MiB RAM detected, ~413556 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DCTN1.fam.
478 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DCTN1.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End

In [137]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'EUR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/DCTN1 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/DCTN1.txt \
    --recode A \
    --out {WORK_DIR}/DCTN1

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/EUR/DCTN1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/EUR/DCTN1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/DCTN1.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/EUR/EUR.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/EUR/DCTN1
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
1 variant loaded from .bim file.
71402 people (39808 males, 31594 females) loaded from .fam.
48340 phenotype values loaded from .fam.
--extract: 1 variant remaining.
--keep: 66894 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 66894 founders and 0 nonfounders present.
Calculating allele frequencies... done.
Total genotyping rate in remainin

In [139]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# DCTN1
recode = pd.read_csv(f'{WORK_DIR}/EUR/DCTN1.raw', sep='\s+')
recode

In [141]:
recode_DCTN1 = recode.copy()
# Define the list of DCTN1 variant column names
DCTN1_variants = [
    'chr2:74363626:C:A_A'
]
# Add the DCTN1_status column
recode_DCTN1['DCTN1_status'] = recode_DCTN1[DCTN1_variants].apply(
    lambda row: 'DCTN1carriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [142]:
# Filter for DCTN1 risk carriers
DCTN1_carriers = recode_DCTN1[recode_DCTN1['DCTN1_status'] == 'DCTN1carriers']
# Count PHENOTYPE values
DCTN1_carriers['PHENOTYPE'].value_counts()

PHENOTYPE
2    4
1    2
Name: count, dtype: int64

In [143]:
# save sample ID of risk carriers
samples_toKeep = DCTN1_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/EUR/DCTN1.samplestoKeep.txt', sep = '\t', index=False, header=None)

FBXO7 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract FBX07 gene in NBA cohort

FBXO7 coordinates: Chromosome 22: 32,474,676-32,498,829 (GRCh38/hg38)

In [144]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr22_{ancestry}_release11_vwb \
    --chr 22 \
    --from-bp 32474676 \
    --to-bp 32498829 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_FBXO7

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_FBXO7.log.
Options in effect:
  --chr 22
  --from-bp 32474676
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_FBXO7
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr22_CAS_release11_vwb
  --to-bp 32498829

Start time: Fri Jan  9 12:14:52 2026
419000 MiB RAM detected, ~414040 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr22_CAS_release11_vwb.psam.
645183 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr22_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 con

In [145]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_FBXO7 \
    --extract ~/workspace/ws_files/cohort/variants.list/FBXO7.txt \
    --make-bed \
    --out {WORK_DIR}/FBXO7

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/FBXO7.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_FBXO7
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/FBXO7.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/FBXO7

Start time: Fri Jan  9 12:15:58 2026
419000 MiB RAM detected, ~414050 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_FBXO7.fam.
479 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_FBXO7.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End

In [146]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'AAC','AFR','EUR','EAS'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/FBXO7 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/FBXO7.txt \
    --recode A \
    --out {WORK_DIR}/FBXO7

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/AAC/FBXO7.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/AAC/FBXO7
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/FBXO7.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/AAC/AAC.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/AAC/FBXO7
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
1 variant loaded from .bim file.
1382 people (627 males, 755 females) loaded from .fam.
1294 phenotype values loaded from .fam.
--extract: 1 variant remaining.
--keep: 1352 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 1352 founders and 0 nonfounders present.
Calculating allele frequencies... done.
1 variant and 1352 people pass filters an

In [147]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# FBXO7
recode = pd.read_csv(f'{WORK_DIR}/AFR/FBXO7.raw', sep='\s+')
recode

In [160]:
recode_FBX = recode.copy()
# Define the list of FBXO7 variant column names
FBX_variants = [
    'chr22:32498165:C:CAA_CAA'
]
# Add the FBX_status column
recode_FBX['FBX_status'] = recode_FBX[FBX_variants].apply(
    lambda row: 'FBXcarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [161]:
# Filter for FBX risk carriers
FBX_carriers = recode_FBX[recode_FBX['FBX_status'] == 'FBXcarriers']
# Count PHENOTYPE values
FBX_carriers['PHENOTYPE'].value_counts()

PHENOTYPE
1    1
Name: count, dtype: int64

In [162]:
# save sample ID of risk carriers
samples_toKeep = ATP_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/FBX.samplestoKeep.txt', sep = '\t', index=False, header=None)

JAM2 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract JAM2 gene in NBA cohort

JAM2 coordinates: Chromosome 21: 25,639,258-25,717,562  (GRCh38/hg38)

In [163]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr21_{ancestry}_release11_vwb \
    --chr 21 \
    --from-bp 25639258 \
    --to-bp 25717562 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_JAM2

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_JAM2.log.
Options in effect:
  --chr 21
  --from-bp 25639258
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_JAM2
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr21_CAS_release11_vwb
  --to-bp 25717562

Start time: Fri Jan  9 12:24:42 2026
419000 MiB RAM detected, ~413994 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr21_CAS_release11_vwb.psam.
621746 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr21_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 contr

In [164]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_JAM2 \
    --extract ~/workspace/ws_files/cohort/variants.list/JAM2.txt \
    --make-bed \
    --out {WORK_DIR}/JAM2

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/JAM2.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_JAM2
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/JAM2.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/JAM2

Start time: Fri Jan  9 12:25:37 2026
419000 MiB RAM detected, ~414017 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_JAM2.fam.
1215 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_JAM2.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End time

RAB39B cohort definition

Annotation of the gene

Extract the region using PLINK

Extract RAB39B gene in NBA cohort

RAB39B coordinates: Chromosome X: 155,258,235-155,264,491 (GRCh38/hg38)

In [165]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chrX_{ancestry}_release11_vwb \
    --chr X \
    --from-bp 155258235 \
    --to-bp 155264491 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_RAB39B

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB39B.log.
Options in effect:
  --chr X
  --from-bp 155258235
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB39B
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chrX_CAS_release11_vwb
  --to-bp 155264491

Start time: Fri Jan  9 12:28:12 2026
419000 MiB RAM detected, ~414034 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chrX_CAS_release11_vwb.psam.
1848687 out of 1916282 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chrX_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053

In [166]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_RAB39B \
    --extract ~/workspace/ws_files/cohort/variants.list/RAB39B.txt \
    --make-bed \
    --out {WORK_DIR}/RAB39B

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/RAB39B.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB39B
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/RAB39B.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/RAB39B

Start time: Fri Jan  9 12:29:29 2026
419000 MiB RAM detected, ~413974 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB39B.fam.
45 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_RAB39B.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters

SLC20A2 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract SLC20A2 gene in NBA cohort

SLC20A2 coordinates: Chromosome 8: 42,416,475-42,541,926 (GRCh38/hg38)

In [167]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr8_{ancestry}_release11_vwb \
    --chr 8 \
    --from-bp 42416475 \
    --to-bp 42541926 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_SLC20A2

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SLC20A2.log.
Options in effect:
  --chr 8
  --from-bp 42416475
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SLC20A2
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr8_CAS_release11_vwb
  --to-bp 42541926

Start time: Fri Jan  9 12:32:12 2026
419000 MiB RAM detected, ~414021 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr8_CAS_release11_vwb.psam.
2641141 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr8_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 co

In [168]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_SLC20A2 \
    --extract ~/workspace/ws_files/cohort/variants.list/SLC20A2.txt \
    --make-bed \
    --out {WORK_DIR}/SLC20A2

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/SLC20A2.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SLC20A2
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/SLC20A2.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/SLC20A2

Start time: Fri Jan  9 12:34:45 2026
419000 MiB RAM detected, ~414002 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SLC20A2.fam.
1926 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SLC20A2.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main

SYNJ1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract SYNJ1 gene in NBA cohort

SYNJ1 coordinates: Chromosome 21: 32,628,759-32,728,040 (GRCh38/hg38)

In [169]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr21_{ancestry}_release11_vwb \
    --chr 21 \
    --from-bp 32628759 \
    --to-bp 32728040 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_SYNJ1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SYNJ1.log.
Options in effect:
  --chr 21
  --from-bp 32628759
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SYNJ1
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr21_CAS_release11_vwb
  --to-bp 32728040

Start time: Fri Jan  9 12:36:54 2026
419000 MiB RAM detected, ~413968 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr21_CAS_release11_vwb.psam.
621746 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr21_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 con

In [170]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_SYNJ1 \
    --extract ~/workspace/ws_files/cohort/variants.list/SYNJ1.txt \
    --make-bed \
    --out {WORK_DIR}/SYNJ1

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/SYNJ1.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SYNJ1
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/SYNJ1.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/SYNJ1

Start time: Fri Jan  9 12:38:16 2026
419000 MiB RAM detected, ~413980 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SYNJ1.fam.
1327 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_SYNJ1.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
En

VPS13C
VPS13C cohort definition

Annotation of the gene

Extract the region using PLINK

Extract VPS13C gene in NBA cohort

VPS13C coordinates: Chromosome 15: 61,852,389-62,060,473 (GRCh38/hg38)

In [171]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr15_{ancestry}_release11_vwb \
    --chr 15 \
    --from-bp 61852389 \
    --to-bp 62060473 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_VPS13C

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS13C.log.
Options in effect:
  --chr 15
  --from-bp 61852389
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS13C
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr15_CAS_release11_vwb
  --to-bp 62060473

Start time: Fri Jan  9 12:39:51 2026
419000 MiB RAM detected, ~413974 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr15_CAS_release11_vwb.psam.
1386456 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr15_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 

In [172]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_VPS13C \
    --extract ~/workspace/ws_files/cohort/variants.list/VPS13C.txt \
    --make-bed \
    --out {WORK_DIR}/VPS13C

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/VPS13C.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS13C
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/VPS13C.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/VPS13C

Start time: Fri Jan  9 12:41:03 2026
419000 MiB RAM detected, ~413901 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS13C.fam.
3775 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_VPS13C.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filte

In [173]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'AAC','AMR','SAS','CAH','EUR','EAS','MDE'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/VPS13C \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/VPS13C.txt \
    --recode A \
    --out {WORK_DIR}/VPS13C

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/AAC/VPS13C.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/AAC/VPS13C
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/VPS13C.txt
  --keep /home/jupyter/workspace/ws_files/r11/cohort/AAC/AAC.samplestoKeep
  --out /home/jupyter/workspace/ws_files/r11/cohort/AAC/VPS13C
  --recode A

419000 MB RAM detected; reserving 209500 MB for main workspace.
1 variant loaded from .bim file.
1382 people (627 males, 755 females) loaded from .fam.
1294 phenotype values loaded from .fam.
--extract: 1 variant remaining.
--keep: 1352 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 1352 founders and 0 nonfounders present.
Calculating allele frequencies... done.
1 variant and 1352 people pass filter

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'


# VPS13C
recode = pd.read_csv(f'{WORK_DIR}/MDE/VPS13C.raw', sep='\s+')
recode

In [197]:
recode_VPS13C = recode.copy()
# Define the list of VPS13C variant column names
VPS13C_variants = [
    'chr15:61873386:G:A_A'
]
# Add the VPS13C_status column
recode_VPS13C['VPS13C_status'] = recode_VPS13C[VPS13C_variants].apply(
    lambda row: 'VPS13Ccarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [198]:
# Filter for VPS13C risk carriers
VPS13C_carriers = recode_VPS13C[recode_VPS13C['VPS13C_status'] == 'VPS13Ccarriers']
# Count PHENOTYPE values
VPS13C_carriers['PHENOTYPE'].value_counts()

Series([], Name: count, dtype: int64)

In [192]:
# save sample ID of risk carriers
samples_toKeep = VPS13C_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AAC/VPS13C.samplestoKeep.txt', sep = '\t', index=False, header=None)

WDR45 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract WDR45 gene in NBA cohort

WDR45 coordinates: Chromosome X: 49,074,433-49,101,170 (GRCh38/hg38)

In [199]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chrX_{ancestry}_release11_vwb \
    --chr X \
    --from-bp 49074433 \
    --to-bp 49101170 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_WDR45

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_WDR45.log.
Options in effect:
  --chr X
  --from-bp 49074433
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_WDR45
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chrX_CAS_release11_vwb
  --to-bp 49101170

Start time: Fri Jan  9 12:56:46 2026
419000 MiB RAM detected, ~413975 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chrX_CAS_release11_vwb.psam.
1848687 out of 1916282 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chrX_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cas

In [200]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_WDR45 \
    --extract ~/workspace/ws_files/cohort/variants.list/WDR45.txt \
    --make-bed \
    --out {WORK_DIR}/WDR45

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/WDR45.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_WDR45
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/WDR45.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/WDR45

Start time: Fri Jan  9 12:57:37 2026
419000 MiB RAM detected, ~413962 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_WDR45.fam.
228 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_WDR45.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filters.
End

DNAJC6 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract DNAJC6 gene in NBA cohort

DNAJC6 coordinates: Chromosome 1: 65,248,219-65,415,871 (GRCh38/hg38)

In [201]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 65248219 \
    --to-bp 65415871 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_DNAJC6

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DNAJC6.log.
Options in effect:
  --chr 1
  --from-bp 65248219
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DNAJC6
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb
  --to-bp 65415871

Start time: Fri Jan  9 12:59:29 2026
419000 MiB RAM detected, ~413958 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.psam.
3776191 variants loaded from
/home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/CAS/chr1_CAS_release11_vwb.pvar.
1 binary phenotype loaded (1053 cases, 1371 cont

In [202]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_DNAJC6 \
    --extract ~/workspace/ws_files/cohort/variants.list/DNAJC6.txt \
    --make-bed \
    --out {WORK_DIR}/DNAJC6

PLINK v2.0.0-a.7LM 64-bit Intel (6 Aug 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/r11/cohort/CAS/DNAJC6.log.
Options in effect:
  --bfile /home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DNAJC6
  --extract /home/jupyter/workspace/ws_files/cohort/variants.list/DNAJC6.txt
  --make-bed
  --out /home/jupyter/workspace/ws_files/r11/cohort/CAS/DNAJC6

Start time: Fri Jan  9 13:00:32 2026
419000 MiB RAM detected, ~413947 available; reserving 209500 MiB for main
workspace.
Using up to 64 threads (change this with --threads).
2801 samples (1602 females, 1199 males; 2801 founders) loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DNAJC6.fam.
2440 variants loaded from
/home/jupyter/workspace/ws_files/r11/cohort/CAS/CAS_DNAJC6.bim.
1 binary phenotype loaded (1053 cases, 1371 controls).
--extract: 0 variants remaining.
Error: No variants remaining after main filte

Cohort sorting

In [18]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# Find IPD in EUR
GBA_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
DCTN1_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/DCTN1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
FBX_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/FBX.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 8 DataFrames
merge = pd.concat([GBA_EUR, LRRK2_EUR, PRKN_EUR, PINK1_EUR, VPS13C_EUR, ATP_EUR, DCTN1_EUR, FBX_EUR], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/EUR.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_EUR = EUR[~EUR['IID'].isin(merge['IID'])]
IPD_EUR

In [21]:
IPD_EUR.to_csv(f'~/workspace/ws_files/r11/cohort/EUR/IPD_EUR_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in AAC
GBA_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
FBXO7_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/FBX.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])


# Merge all 4 DataFrames
merge = pd.concat([GBA_AAC, LRRK2_AAC, PRKN_AAC, FBXO7_AAC, VPS13C_AAC], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/AAC.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_AAC = AAC[~AAC['IID'].isin(merge['IID'])]
IPD_AAC

In [24]:
IPD_AAC.to_csv(f'~/workspace/ws_files/r11/cohort/AAC/IPD_AAC_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/AFR'
# Find IPD in AFR
GBA_AFR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AFR/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AFR = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
DJ1_AFR = pd.read_csv(f'{WORK_DIR}/DJ1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_AFR = pd.read_csv(f'{WORK_DIR}/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AFR = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_AFR = pd.read_csv(f'{WORK_DIR}/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
FBX_AFR = pd.read_csv(f'{WORK_DIR}/FBX.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 6 DataFrames
merge = pd.concat([GBA_AFR, LRRK2_AFR, DJ1_AFR, PINK1_AFR, PRKN_AFR, ATP_AFR, FBX_AFR], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AFR = pd.read_csv(f'{WORK_DIR}/AFR.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_AFR = AFR[~AFR['IID'].isin(merge['IID'])]
IPD_AFR

In [26]:
IPD_AFR.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/IPD_AFR_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/AJ'
# Find IPD in AJ
GBA_AJ = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AJ/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AJ = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AJ = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_AJ = pd.read_csv(f'{WORK_DIR}/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 4 DataFrames
merge = pd.concat([GBA_AJ, PRKN_AJ, LRRK2_AJ, ATP_AJ], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AJ = pd.read_csv(f'{WORK_DIR}/AJ.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_AJ = AJ[~AJ['IID'].isin(merge['IID'])]
IPD_AJ

In [28]:
IPD_AJ.to_csv(f'~/workspace/ws_files/r11/cohort/AJ/IPD_AJ_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/SAS'
# Find IPD in AJ
GBA_SAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/SAS/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_SAS = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_SAS = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
DJ1_SAS = pd.read_csv(f'{WORK_DIR}/DJ1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 3 DataFrames
merge = pd.concat([GBA_SAS, LRRK2_SAS, PRKN_SAS, DJ1_SAS], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

SAS = pd.read_csv(f'{WORK_DIR}/SAS.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_SAS = SAS[~SAS['IID'].isin(merge['IID'])]
IPD_SAS

In [30]:
IPD_SAS.to_csv(f'~/workspace/ws_files/r11/cohort/SAS/IPD_SAS_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/AMR'
# Find IPD in AMR
GBA_AMR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AMR/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AMR = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AMR = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_AMR = pd.read_csv(f'{WORK_DIR}/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_AMR = pd.read_csv(f'{WORK_DIR}/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_AMR = pd.read_csv(f'{WORK_DIR}/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 6 DataFrames
merge = pd.concat([GBA_AMR, PRKN_AMR, LRRK2_AMR, PINK1_AMR, VPS13C_AMR, ATP_AMR], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AMR = pd.read_csv(f'{WORK_DIR}/AMR.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter AMR DataFrame to exclude IIDs present in merged_ipd
IPD_AMR = AMR[~AMR['IID'].isin(merge['IID'])]
IPD_AMR

In [32]:
IPD_AMR.to_csv(f'~/workspace/ws_files/r11/cohort/AMR/IPD_AMR_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in EAS
GBA_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
DJ1_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/DJ1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])


# Merge all 5 DataFrames
merge = pd.concat([GBA_EAS, LRRK2_EAS, PRKN_EAS, PINK1_EAS, DJ1_EAS], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/EAS.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EAS DataFrame to exclude IIDs present in merged_ipd
IPD_EAS = EAS[~EAS['IID'].isin(merge['IID'])]
IPD_EAS

In [34]:
IPD_EAS.to_csv(f'~/workspace/ws_files/r11/cohort/EAS/IPD_EAS_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in CAS
GBA_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 4 DataFrames
merge = pd.concat([GBA_CAS, LRRK2_CAS, PRKN_CAS, PINK1_CAS], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/CAS.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter CAS DataFrame to exclude IIDs present in merged_ipd
IPD_CAS = CAS[~CAS['IID'].isin(merge['IID'])]
IPD_CAS

In [36]:
IPD_CAS.to_csv(f'~/workspace/ws_files/r11/cohort/CAS/IPD_CAS_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in MDE
GBA_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 4 DataFrames
merge = pd.concat([GBA_MDE, LRRK2_MDE, PRKN_MDE, PINK1_MDE], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/MDE.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter MDE DataFrame to exclude IIDs present in merged_ipd
IPD_MDE = MDE[~MDE['IID'].isin(merge['IID'])]
IPD_MDE

In [38]:
IPD_MDE.to_csv(f'~/workspace/ws_files/r11/cohort/MDE/IPD_MDE_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in CAH
GBA_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])


# Merge all 4 DataFrames
merge = pd.concat([GBA_CAH, LRRK2_CAH, PRKN_CAH, ATP_CAH, VPS13C_CAH], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/CAH.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter CAH DataFrame to exclude IIDs present in merged_ipd
IPD_CAH = CAH[~CAH['IID'].isin(merge['IID'])]
IPD_CAH

In [246]:
IPD_CAH.to_csv(f'~/workspace/ws_files/r11/cohort/CAH/IPD_CAH_rm.txt', sep = '\t', index=False, header=None)